# TensorRT 推理教程

本教程详细介绍 NVIDIA TensorRT 推理引擎的使用，包括：

1. **TensorRT 基础**: 核心概念和优化技术
2. **引擎构建**: 从 ONNX 构建 TensorRT 引擎
3. **精度模式**: FP32/FP16/INT8
4. **动态形状**: 支持可变输入大小

---

## 什么是 TensorRT？

TensorRT 是 NVIDIA 的高性能深度学习推理优化器和运行时：

```
ONNX/PyTorch Model
        │
        ▼
┌───────────────────┐
│   TensorRT 优化    │
│  ┌─────────────┐  │
│  │  层融合      │  │
│  │  精度校准    │  │
│  │  内核自动调优 │  │
│  │  内存优化    │  │
│  └─────────────┘  │
└───────────────────┘
        │
        ▼
   TensorRT Engine
   (序列化的优化模型)
```

**注意**: TensorRT 仅支持 NVIDIA GPU

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import os
import tempfile

# 检查 TensorRT
try:
    import tensorrt as trt
    print(f"TensorRT 版本: {trt.__version__}")
    TENSORRT_AVAILABLE = True
except ImportError:
    print("TensorRT 未安装")
    print("安装命令: pip install tensorrt")
    TENSORRT_AVAILABLE = False

# 检查 PyCUDA
try:
    import pycuda.driver as cuda
    import pycuda.autoinit
    print(f"PyCUDA 可用")
    PYCUDA_AVAILABLE = True
except ImportError:
    print("PyCUDA 未安装")
    print("安装命令: pip install pycuda")
    PYCUDA_AVAILABLE = False

# 检查 PyTorch
try:
    import torch
    import torch.nn as nn
    print(f"PyTorch 版本: {torch.__version__}")
    print(f"CUDA 可用: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}")
except ImportError:
    print("PyTorch 未安装")

## 1. TensorRT 核心优化技术

### 1.1 层融合 (Layer Fusion)

```
优化前:                    优化后:
┌─────┐                   ┌─────────────┐
│Conv │                   │             │
└──┬──┘                   │ Conv+BN+ReLU│
   │                      │  (融合内核)  │
┌──▼──┐                   │             │
│ BN  │         →         └─────────────┘
└──┬──┘
   │
┌──▼──┐
│ReLU │
└─────┘

内存访问: 3次 → 1次
内核启动: 3次 → 1次
```

### 1.2 精度模式

| 精度 | 位数 | 速度 | 精度损失 |
|:-----|:----:|:----:|:--------:|
| FP32 | 32 | 1x | 无 |
| FP16 | 16 | 2x | 极小 |
| INT8 | 8 | 4x | 需要校准 |

## 2. 创建测试模型

In [ ]:
# 定义测试模型
class SimpleConvNet(nn.Module):
    """简单卷积网络"""
    def __init__(self, num_classes=10):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 32, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.pool = nn.MaxPool2d(2)
        self.fc1 = nn.Linear(64 * 8 * 8, 256)
        self.fc2 = nn.Linear(256, num_classes)
    
    def forward(self, x):
        x = self.pool(torch.relu(self.bn1(self.conv1(x))))
        x = self.pool(torch.relu(self.bn2(self.conv2(x))))
        x = x.view(x.size(0), -1)
        x = torch.relu(self.fc1(x))
        return self.fc2(x)

# 创建模型
model = SimpleConvNet()
model.eval()

print(f"模型参数量: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# 导出为 ONNX
model_dir = tempfile.mkdtemp()
onnx_path = os.path.join(model_dir, "model.onnx")

dummy_input = torch.randn(1, 3, 32, 32)

torch.onnx.export(
    model,
    dummy_input,
    onnx_path,
    input_names=['input'],
    output_names=['output'],
    dynamic_axes={
        'input': {0: 'batch_size'},
        'output': {0: 'batch_size'}
    },
    opset_version=14
)

print(f"ONNX 模型已保存: {onnx_path}")
print(f"模型大小: {os.path.getsize(onnx_path) / 1024:.2f} KB")

## 3. 构建 TensorRT 引擎

以下代码需要 TensorRT 和 NVIDIA GPU 才能运行。

In [ ]:
if TENSORRT_AVAILABLE:
    # 创建 Logger
    logger = trt.Logger(trt.Logger.WARNING)
    
    # 创建 Builder
    builder = trt.Builder(logger)
    
    # 创建网络 (显式批次)
    network_flags = 1 << int(trt.NetworkDefinitionCreationFlag.EXPLICIT_BATCH)
    network = builder.create_network(network_flags)
    
    # 解析 ONNX
    parser = trt.OnnxParser(network, logger)
    with open(onnx_path, "rb") as f:
        if parser.parse(f.read()):
            print("ONNX 解析成功!")
            print(f"网络输入数: {network.num_inputs}")
            print(f"网络输出数: {network.num_outputs}")
            print(f"网络层数: {network.num_layers}")
        else:
            print("ONNX 解析失败!")
            for i in range(parser.num_errors):
                print(f"  错误: {parser.get_error(i)}")
else:
    print("跳过 TensorRT 示例 (TensorRT 未安装)")

In [ ]:
if TENSORRT_AVAILABLE:
    # 配置构建选项
    config = builder.create_builder_config()
    
    # 设置工作空间大小 (1GB)
    config.set_memory_pool_limit(trt.MemoryPoolType.WORKSPACE, 1 << 30)
    
    # 启用 FP16 (如果支持)
    if builder.platform_has_fast_fp16:
        config.set_flag(trt.BuilderFlag.FP16)
        print("启用 FP16 精度")
    
    # 配置动态形状
    profile = builder.create_optimization_profile()
    profile.set_shape(
        "input",
        min=(1, 3, 32, 32),    # 最小形状
        opt=(8, 3, 32, 32),    # 最优形状
        max=(32, 3, 32, 32)    # 最大形状
    )
    config.add_optimization_profile(profile)
    
    print("构建配置完成!")
else:
    print("跳过 TensorRT 示例")

In [ ]:
if TENSORRT_AVAILABLE:
    # 构建引擎
    print("正在构建 TensorRT 引擎 (可能需要几分钟)...")
    serialized_engine = builder.build_serialized_network(network, config)
    
    if serialized_engine:
        print(f"引擎构建成功!")
        print(f"引擎大小: {len(serialized_engine) / 1024:.2f} KB")
        
        # 保存引擎
        engine_path = os.path.join(model_dir, "model.engine")
        with open(engine_path, "wb") as f:
            f.write(serialized_engine)
        print(f"引擎已保存: {engine_path}")
    else:
        print("引擎构建失败!")
else:
    print("跳过 TensorRT 示例")

## 4. 使用封装的模块

In [ ]:
from tensorrt_engine import (
    EngineConfig,
    Precision,
    CalibrationAlgorithm,
    TENSORRT_AVAILABLE,
    PYCUDA_AVAILABLE
)

# 创建配置
config = EngineConfig(
    precision=Precision.FP16,
    workspace_size=1 << 30,  # 1GB
    min_batch_size=1,
    opt_batch_size=8,
    max_batch_size=32
)

print("引擎配置:")
print(f"  精度: {config.precision.value}")
print(f"  工作空间: {config.workspace_size / (1024**3):.1f} GB")
print(f"  批次范围: {config.min_batch_size} - {config.max_batch_size}")

In [ ]:
if TENSORRT_AVAILABLE and PYCUDA_AVAILABLE:
    from tensorrt_engine import EngineBuilder, TensorRTEngine
    
    # 使用 EngineBuilder 构建引擎
    builder = EngineBuilder(config)
    
    engine_path = os.path.join(model_dir, "model_v2.engine")
    
    try:
        serialized = builder.build_from_onnx(
            onnx_path,
            engine_path,
            input_shapes={
                'input': (
                    (1, 3, 32, 32),   # min
                    (8, 3, 32, 32),   # opt
                    (32, 3, 32, 32)   # max
                )
            }
        )
        print(f"引擎构建成功: {engine_path}")
    except Exception as e:
        print(f"引擎构建失败: {e}")
else:
    print("跳过 TensorRT 示例 (依赖未安装)")

## 5. TensorRT 推理

In [ ]:
if TENSORRT_AVAILABLE and PYCUDA_AVAILABLE and os.path.exists(engine_path):
    # 加载引擎
    engine = TensorRTEngine(engine_path)
    
    print(f"输入名称: {engine.input_names}")
    print(f"输出名称: {engine.output_names}")
    
    # 执行推理
    input_data = np.random.randn(1, 3, 32, 32).astype(np.float32)
    outputs = engine.infer({'input': input_data})
    
    print(f"\n输出形状: {outputs['output'].shape}")
    print(f"输出示例: {outputs['output'][0][:5]}")
else:
    print("跳过 TensorRT 推理示例")

## 6. 性能对比: PyTorch vs TensorRT

In [ ]:
import time

def benchmark_pytorch(model, input_data, num_runs=100, warmup=10):
    """PyTorch 基准测试"""
    model.eval()
    
    # 预热
    with torch.no_grad():
        for _ in range(warmup):
            _ = model(input_data)
    
    # 计时
    latencies = []
    with torch.no_grad():
        for _ in range(num_runs):
            start = time.perf_counter()
            _ = model(input_data)
            latencies.append((time.perf_counter() - start) * 1000)
    
    return np.mean(latencies), np.std(latencies)

# PyTorch CPU 基准
input_tensor = torch.randn(8, 3, 32, 32)
pytorch_mean, pytorch_std = benchmark_pytorch(model, input_tensor)
print(f"PyTorch CPU: {pytorch_mean:.2f} ± {pytorch_std:.2f} ms")

# PyTorch GPU 基准 (如果可用)
if torch.cuda.is_available():
    model_gpu = model.cuda()
    input_gpu = input_tensor.cuda()
    
    # 同步 GPU
    torch.cuda.synchronize()
    pytorch_gpu_mean, pytorch_gpu_std = benchmark_pytorch(model_gpu, input_gpu)
    torch.cuda.synchronize()
    
    print(f"PyTorch GPU: {pytorch_gpu_mean:.2f} ± {pytorch_gpu_std:.2f} ms")

In [ ]:
if TENSORRT_AVAILABLE and PYCUDA_AVAILABLE and 'engine' in dir():
    def benchmark_tensorrt(engine, input_data, num_runs=100, warmup=10):
        """TensorRT 基准测试"""
        # 预热
        for _ in range(warmup):
            _ = engine.infer({'input': input_data})
        
        # 计时
        latencies = []
        for _ in range(num_runs):
            start = time.perf_counter()
            _ = engine.infer({'input': input_data})
            latencies.append((time.perf_counter() - start) * 1000)
        
        return np.mean(latencies), np.std(latencies)
    
    # TensorRT 基准
    input_np = np.random.randn(8, 3, 32, 32).astype(np.float32)
    trt_mean, trt_std = benchmark_tensorrt(engine, input_np)
    print(f"TensorRT: {trt_mean:.2f} ± {trt_std:.2f} ms")
    
    # 计算加速比
    if torch.cuda.is_available():
        speedup = pytorch_gpu_mean / trt_mean
        print(f"\nTensorRT vs PyTorch GPU 加速比: {speedup:.2f}x")
else:
    print("跳过 TensorRT 基准测试")

## 7. INT8 量化 (需要校准数据)

In [ ]:
# INT8 量化配置示例
int8_config = EngineConfig(
    precision=Precision.INT8,
    workspace_size=1 << 30,
    calibration_algorithm=CalibrationAlgorithm.ENTROPY,
    # 校准数据需要在实际使用时提供
    # calibration_data=[...],
    # calibration_cache_file="calibration.cache"
)

print("INT8 量化配置:")
print(f"  精度: {int8_config.precision.value}")
print(f"  校准算法: {int8_config.calibration_algorithm.value}")
print("\n注意: INT8 量化需要提供校准数据集")

## 总结

本教程介绍了 TensorRT 的核心功能：

1. **核心优化**: 层融合、精度校准、内核自动调优
2. **引擎构建**: 从 ONNX 构建优化引擎
3. **精度模式**: FP32/FP16/INT8
4. **动态形状**: 支持可变批次大小

### 最佳实践

- 使用 FP16 获得 2x 加速，精度损失极小
- INT8 需要校准数据，可获得 4x 加速
- 配置合适的动态形状范围
- 保存引擎文件避免重复构建

In [ ]:
# 清理临时文件
import shutil
shutil.rmtree(model_dir, ignore_errors=True)
print("临时文件已清理")